# nb_01 — Bronze: ingest reference masters & the pay-grid feed

**Module 1 (notebook half).** The pipeline handles the flat monthly workforce-event
extracts. This notebook handles the sources a Copy activity handles badly: the
**nested JSON pay-grid feed** (arrays of band history) plus FX and the reference
masters.

**Prereqs**
- Schema-enabled lakehouse `lh_meridian_hr` attached as the default lakehouse.
- `BASE` below points at your fork's raw GitHub `/data` folder.

Everything lands in the `bronze` schema — a faithful copy of the source.

## Configuration and Spark setup

**Summary.** Sets the raw source base URL, the landing folder, and the lakehouse name; enables V-Order for Direct Lake; and ensures the `bronze` schema exists.

<details>
<summary>Line-by-line details</summary>

- `BASE` — the raw GitHub `/data` folder every feed is downloaded from.
- `LANDING = "Files/landing"` — the lakehouse-relative folder the feeds land in.
- `LH = "lh_meridian_hr"` — the target lakehouse name.
- `from pyspark.sql import functions as F` — the `F.*` helpers used throughout.
- `spark.conf.set("spark.sql.parquet.vorder.default", "true")` — turn on V-Order so the Delta files are optimized for Direct Lake reads.
- `CREATE SCHEMA IF NOT EXISTS bronze` — ensure the Bronze schema exists.

</details>

In [ ]:
BASE = "https://raw.githubusercontent.com/modamin/fabric-developer-training-lab/main/data"
LANDING = "Files/landing"
LH = "lh_meridian_hr"

from pyspark.sql import functions as F
spark.conf.set("spark.sql.parquet.vorder.default", "true")   # V-Order for Direct Lake
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
print("landing feeds from:", BASE)

## 1. Download the feeds into `Files/landing`

Download the hosted files from the raw GitHub repository into the lakehouse file
area. In production, the pipeline lands these files.

## Download the feeds into `Files/landing`

**Summary.** Downloads each hosted feed into the lakehouse file area, guarding against HTML error pages and validating JSON before writing.

<details>
<summary>Line-by-line details</summary>

- `LOCAL = "/lakehouse/default/Files/landing"` + `os.makedirs(..., exist_ok=True)` — the local mount path for the landing folder, created if needed.
- `def fetch(relative_path)` — downloads one file: builds `source_url` from `BASE`, opens it with `urllib.request.urlopen`, and raises a clear error if the request fails.
- The `if b"<html" ...` check rejects HTML (a sign of a non-raw URL); `if relative_path.endswith(".json"): json.loads(content)` validates JSON before saving.
- `with open(destination, "wb") ...` — write the bytes to the landing folder and print the size.
- The `for relative_path in [...]` loop fetches the pay-grid feed, FX rates, and the three reference masters.

</details>

In [ ]:
import json
import os
import urllib.request

LOCAL = "/lakehouse/default/Files/landing"
os.makedirs(LOCAL, exist_ok=True)

def fetch(relative_path):
    source_url = f"{BASE}/{relative_path}"
    destination = f"{LOCAL}/{os.path.basename(relative_path)}"
    try:
        with urllib.request.urlopen(source_url) as response:
            content = response.read()
    except Exception as error:
        raise RuntimeError(f"Could not download {source_url}. Check BASE and repository access.") from error

    if b"<html" in content[:1000].lower() or b"<!doctype html" in content[:1000].lower():
        raise ValueError(f"Downloaded HTML instead of data from {source_url}. Use a raw GitHub URL.")
    if relative_path.endswith(".json"):
        json.loads(content)

    with open(destination, "wb") as output_file:
        output_file.write(content)
    print("  ", destination, len(content), "bytes")

for relative_path in [
    "feeds/pay_bands_feed.json",
    "feeds/fx_rates.csv",
    "reference/workers.csv",
    "reference/workers_delta.csv",
    "reference/cost_centers.csv",
]:
    fetch(relative_path)

## 2. Pay-grid feed — explode the nested band history

One object per (classification group, level) with a `band_history` array. We read
it as multiline JSON and `explode` the array into one row per (group, level,
effective_date). This Bronze table is the **source for the SCD Type 2 pay-band
dimension** in Module 3.

## Pay-grid feed — explode the nested band history

**Summary.** Reads the multiline JSON pay-grid feed and explodes its nested arrays into one flat row per (group, level, effective date), writing `bronze.pay_bands` — the source for the SCD2 pay-band dimension.

<details>
<summary>Line-by-line details</summary>

- `spark.read.option("multiLine", "true").json(pay_bands_path)` — read the feed as multiline JSON.
- The `if "classifications" not in raw.columns` guard fails early if the file shape is wrong (usually a bad `BASE`).
- `F.explode("classifications")` — one row per classification object.
- The second `.select(...)` pulls `group` and `level`, then `F.explode("c.band_history")` fans out to one row per historical band.
- The final `.select(...)` casts `effective_date`, `band_min/mid/max`, and stamps `_ingested_at` with `F.current_timestamp()`.
- `write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("bronze.pay_bands")` — overwrite the Bronze table.
- The `print` and `filter(... PA / level 3 ...).show()` confirm the row count and preview one classification's history.

</details>

In [ ]:
pay_bands_path = f"{LANDING}/pay_bands_feed.json"
raw = (spark.read.option("multiLine", "true")
       .json(pay_bands_path))

if "classifications" not in raw.columns:
    raise ValueError(
        f"Expected 'classifications' in {pay_bands_path}; found {raw.columns}. "
        "Check that BASE points to the raw GitHub data folder."
    )

pay_bands = (raw
    .select(F.explode("classifications").alias("c"))
    .select(F.col("c.group").alias("classification_group"),
            F.col("c.level").cast("int").alias("classification_level"),
            F.explode("c.band_history").alias("h"))
    .select("classification_group","classification_level",
            F.to_date("h.effective_date").alias("band_effective_date"),
            F.col("h.band_min").cast("double").alias("band_min"),
            F.col("h.band_mid").cast("double").alias("band_mid"),
            F.col("h.band_max").cast("double").alias("band_max"),
            F.current_timestamp().alias("_ingested_at")))
(pay_bands.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("bronze.pay_bands"))
print(pay_bands.count(), "band-history rows")
pay_bands.filter("classification_group='PA' AND classification_level=3") \
    .orderBy("band_effective_date").show(truncate=False)

## 3. FX and reference masters (flat CSV → Bronze Delta)

## FX and reference masters (flat CSV to Bronze Delta)

**Summary.** Defines a small helper that lands a header CSV into a Delta table with an ingest timestamp, then applies it to FX rates and the three reference masters.

<details>
<summary>Line-by-line details</summary>

- `def land_csv(path, table)` — reads a CSV with `header=true` and `inferSchema=true`, adds `_ingested_at`, and overwrites the target Delta table (with `overwriteSchema`).
- The `print(...)` reports the table name and row count.
- The four `land_csv(...)` calls land `bronze.fx_rates`, `bronze.workers`, `bronze.workers_delta`, and `bronze.cost_centers`.

</details>

In [ ]:
def land_csv(path, table):
    df = (spark.read.option("header","true").option("inferSchema","true")
          .csv(path).withColumn("_ingested_at", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite").option("overwriteSchema","true")
       .saveAsTable(table))
    print(f"{table:28s} {df.count():>7,} rows")

land_csv(f"{LANDING}/fx_rates.csv",      "bronze.fx_rates")
land_csv(f"{LANDING}/workers.csv",       "bronze.workers")
land_csv(f"{LANDING}/workers_delta.csv", "bronze.workers_delta")
land_csv(f"{LANDING}/cost_centers.csv",  "bronze.cost_centers")

## 4. Confirm the pipeline landed the events

The pipeline runs `nb_00_setup_lakehouse` to create the target table and
`nb_00_setup_watermark` to initialize its last-loaded month. It then appends only the
months after that watermark to `bronze.workforce_events_raw` and advances the
watermark after every Copy succeeds. The fallback below downloads the files from
GitHub only if the target table does not exist.

## Confirm the pipeline landed the events (with fallback)

**Summary.** Checks that the pipeline-loaded `bronze.workforce_events_raw` exists; if it does not, downloads the monthly event CSVs from GitHub and loads them so the rest of the lab can proceed.

<details>
<summary>Line-by-line details</summary>

- `try: ... spark.table('bronze.workforce_events_raw').count()` — prints the event count when the table exists.
- `except AnalysisException:` — the table is missing, so run the fallback loader.
- The `for month in pd.period_range("2021-01", "2025-12", freq="M")` loop downloads each monthly CSV with `urllib.request.urlretrieve`, raising a clear error on failure.
- `spark.read ... csv(f"{LANDING}/events/*.csv")` adds `_source_file` (`input_file_name()`) and `_ingested_at`, then overwrites `bronze.workforce_events_raw`.
- The final `print` reports how many rows the fallback loaded.

</details>

In [ ]:
from pyspark.sql.utils import AnalysisException

try:
    print("pipeline-loaded events:", f"{spark.table('bronze.workforce_events_raw').count():,}")
except AnalysisException:
    print("not found — downloading events from GitHub")
    import pandas as pd

    events_local = "/lakehouse/default/Files/landing/events"
    os.makedirs(events_local, exist_ok=True)
    for month in pd.period_range("2021-01", "2025-12", freq="M").strftime("%Y-%m"):
        source_url = f"{BASE}/events/workforce_events_{month}.csv"
        destination = f"{events_local}/workforce_events_{month}.csv"
        try:
            urllib.request.urlretrieve(source_url, destination)
        except Exception as error:
            raise RuntimeError(f"Could not download {source_url}. Check BASE and repository access.") from error

    raw = (spark.read.option("header","true").option("inferSchema","true")
           .csv(f"{LANDING}/events/*.csv")
           .withColumn("_source_file", F.input_file_name())
           .withColumn("_ingested_at", F.current_timestamp()))
    (raw.write.format("delta").mode("overwrite").option("overwriteSchema","true")
        .saveAsTable("bronze.workforce_events_raw"))
    print("fallback-loaded events:", f"{raw.count():,}")

## List the Bronze tables

**Summary.** Prints the tables now present in the `bronze` schema as a quick completion check.

<details>
<summary>Line-by-line details</summary>

- `spark.sql("SHOW TABLES IN bronze").collect()` — list the Bronze tables and print each `tableName`.

</details>

In [ ]:
for t in spark.sql("SHOW TABLES IN bronze").collect(): print("  ", t.tableName)